# 01 — Dataset

Explore the raw imbalance data, choose a dependent variable, and clean the panel.
Output: `data/clean.parquet`, consumed by 02 and 03.

In [17]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

DATA = Path("data")
paths = sorted(DATA.glob("*.csv.gz"))
len(paths), paths[0].name, paths[-1].name

(21, '20250303.csv.gz', '20250331.csv.gz')

In [18]:
raw = pd.read_csv(paths[0])
raw.info()
raw.describe().T[["count", "min", "50%", "max"]]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195371 entries, 0 to 195370
Data columns (total 17 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   ts             195371 non-null  object 
 1   adv            195371 non-null  float64
 2   ask            195364 non-null  float64
 3   ask_qty        195364 non-null  float64
 4   bid            195364 non-null  float64
 5   bid_qty        195364 non-null  float64
 6   cross          30320 non-null   float64
 7   far_price      194871 non-null  float64
 8   near_price     194871 non-null  float64
 9   open           195371 non-null  float64
 10  paired_shares  194871 non-null  float64
 11  ref_price      194871 non-null  float64
 12  shares         194871 non-null  float64
 13  side           194871 non-null  object 
 14  symbol         195371 non-null  object 
 15  ts.1           195371 non-null  object 
 16  local_time     195371 non-null  object 
dtypes: float64(12), object(5)
mem

,count,min,50%,max
adv,195371.0,39.10,2589.850,2.508206e+05
ask,195364.0,1.20,73.610,4.969000e+03
ask_qty,195364.0,1.00,188.000,6.271790e+05
bid,195364.0,1.19,73.380,4.950150e+03
bid_qty,195364.0,1.00,137.000,5.190160e+05
cross,30320.0,0.00,73.515,4.946150e+03
far_price,194871.0,0.00,46.160,4.941470e+03
near_price,194871.0,0.00,49.310,4.943000e+03
open,195371.0,0.00,75.730,5.016010e+03
paired_shares,194871.0,0.00,133368.000,2.261339e+07


`describe()` flags the two things worth chasing: `cross` is mostly null, and
`cross`, `open`, `ref_price`, `near_price` and `far_price` all have a minimum of
`0.0`, which is impossible for a price.

In [19]:
ts = pd.to_datetime(raw["ts"], utc=True).dt.tz_convert("America/New_York")
m = ts.dt.strftime("%H:%M")

print(f"{raw['symbol'].nunique()} symbols, median {raw.groupby('symbol').size().median():.0f} messages each")
print(f"ts is UTC: {raw['ts'].iloc[0][:19]} = {ts.iloc[0].strftime('%H:%M')} ET\n")
m.value_counts().sort_index()

500 symbols, median 391 messages each
ts is UTC: 2025-03-03 20:50:00 = 15:50 ET



ts
15:50     3000
15:51     3000
15:52     3000
15:53     3000
15:54     3000
15:55    30000
15:56    30000
15:57    30000
15:58    30000
15:59    30000
16:00     6018
16:01     5964
16:02     5964
16:03     5964
16:04     5964
16:05      497
Name: count, dtype: int64

## Where the label lives

`cross` is null on 84.5% of rows. It appears only at 16:00+, is constant per symbol,
and arrives on 500 dedicated print rows — one per symbol, with `side` blank. That is
the auction result message, and the cleanest label source.

In [20]:
print("cross null fraction:", round(raw["cross"].isna().mean(), 3))
print("cross appears only at:", sorted(m[raw["cross"].notna()].unique()))
print("symbols with >1 distinct cross:",
      int((raw[raw["cross"].notna()].groupby("symbol")["cross"].nunique() > 1).sum()))

prints = raw[raw["side"].isna()]
print(f"print rows (side null): {len(prints)} rows / {prints['symbol'].nunique()} symbols")

cross null fraction: 0.845
cross appears only at: ['16:00', '16:01', '16:02', '16:03', '16:04', '16:05']
symbols with >1 distinct cross: 0
print rows (side null): 500 rows / 500 symbols


## `0.0` means missing, and three ETFs have no cross

Every price column uses `0.0` rather than null for an absent price, so `.isna()`
alone is not sufficient. `near_price` and `far_price` are 100% zero at 15:50
because they are auction-only fields not published until 15:55.

The zero-`cross` symbols are USO, IWM and SPY — NYSE Arca-listed ETFs. They trade
on Nasdaq but their closing auction runs at their listing exchange, so no Nasdaq
cross exists. Here `0.0` means "not applicable", and a zero label would imply a
−10,000 bps return on every one of their rows.

In [21]:
print("fraction == 0, by minute:")
print(raw.assign(m=m).groupby("m")[["ref_price", "near_price", "far_price"]]
         .apply(lambda g: (g == 0).mean()).loc[["15:50", "15:55", "15:59"]].round(4))

print("\ncross == 0 :", raw.loc[raw["cross"] == 0, "symbol"].unique())
print("null bid/ask:", raw["bid"].isna().sum(),
      "rows |", raw.loc[raw["bid"].isna(), "symbol"].unique())

fraction == 0, by minute:
       ref_price  near_price  far_price
m                                      
15:50      0.010       1.000     1.0000
15:55      0.008       0.008     0.0942
15:59      0.006       0.006     0.0112

cross == 0 : ['USO' 'IWM' 'SPY']
null bid/ask: 7 rows | ['FERAU']


In [22]:
def load_day(path):
    """Parse one day. Parsing decisions only — no row dropping except post-close."""
    df = pd.read_csv(path).drop(columns=["ts.1"])
    df["ts"] = pd.to_datetime(df["ts"], utc=True).dt.tz_convert("America/New_York")

    # this file encodes "no price" as 0.0 rather than null
    price_cols = ["ref_price", "near_price", "far_price"]
    df[price_cols] = df[price_cols].replace(0, np.nan)

    # cross is published after the auction and is constant per symbol
    post = df.loc[df["cross"].notna()]
    df["cross"] = df["symbol"].map(post.groupby("symbol")["cross"].first())

    close = df["ts"].dt.normalize() + pd.Timedelta(hours=16)
    df["secs_to_close"] = (close - df["ts"]).dt.total_seconds()
    df["date"] = df["ts"].dt.normalize()

    # everything at or after 16:00 is post-auction — the label itself
    return df[df["secs_to_close"] > 0].copy()

In [23]:
panel = pd.concat([load_day(p) for p in paths], ignore_index=True)
panel["symbol"] = panel["symbol"].astype("category")
panel["side"] = panel["side"].astype("category")

print(f"{panel.shape[0]:,} rows | {panel['date'].nunique()} days | {panel['symbol'].nunique()} symbols")
print("symbols per day:", panel.groupby("date")["symbol"].nunique().unique())
print("present all 21 days:", int((panel.groupby("symbol", observed=True)["date"].nunique() == 21).sum()))
print("secs_to_close:", round(panel["secs_to_close"].min(), 2), "-", round(panel["secs_to_close"].max(), 2))

sd = panel.groupby(["symbol", "date"], observed=True)["cross"].first()
bad = sd[sd.isna() | (sd <= 0)]
print(f"\nbad symbol-days: {len(bad)} of {len(sd)} ({len(bad)/len(sd):.2%})")
print(bad.reset_index()["symbol"].astype(str).value_counts().to_string())

3,465,000 rows | 21 days | 562 symbols
symbols per day: [500]
present all 21 days: 443
secs_to_close: 0.82 - 600.0

bad symbol-days: 60 of 10500 (0.57%)
symbol
USO     21
IWM     19
SPY     13
FNGA     2
ALNY     1
ALTR     1
AZPN     1
NVDD     1
RAA      1


## Panel structure and cleaning

3,465,000 rows = 21 days × 500 symbols × 330 messages (30 at 10s intervals from
15:50, then 300 at 1s from 15:55). 562 distinct symbols appear across the month but
exactly 500 per day, so the universe is rebalanced mid-month.

Three cleaning decisions:

- **Bad `cross` (60 of 10,500 symbol-days, 0.57%)** — the three Arca ETFs account for
  most of it, and six other symbols have isolated one- or two-day gaps. Dropping at
  the **symbol-day** level handles both without discarding valid days.
- **Quotes** — a handful of rows have a null or non-positive bid/ask and are dropped.
- **`open`** — only feeds one feature, so a bad value is nulled rather than costing
  the whole row.

`secs_to_close` spanning (0.8, 600.0] with no violations confirms the UTC→ET
conversion survives the 2025-03-09 DST change.

In [24]:
def clean(df):
    n0 = len(df)

    # symbol-days with no usable label
    lab = df.groupby(["symbol", "date"], observed=True)["cross"].first()
    bad = set(lab[lab.isna() | (lab <= 0)].index)
    key = pd.MultiIndex.from_arrays([df["symbol"], df["date"]])
    df = df.loc[~key.isin(bad)]

    # unusable quotes
    df = df.loc[df["bid"].notna() & df["ask"].notna()]
    df = df.loc[(df["bid"] > 0) & (df["ask"] > 0) & (df["ask"] >= df["bid"])].copy()

    # `open` only feeds one feature — null it rather than drop the row
    df.loc[df["open"] <= 0, "open"] = np.nan

    print(f"{n0:,} -> {len(df):,} rows ({len(df)/n0:.2%} kept)")
    return df.reset_index(drop=True)


clean_panel = clean(panel)
nulls = clean_panel.isna().mean()
print("remaining nulls:", nulls[nulls > 0].round(4).to_dict())

3,465,000 -> 3,445,083 rows (99.43% kept)
remaining nulls: {'far_price': 0.1229, 'near_price': 0.0909, 'open': 0.0002, 'ref_price': 0.0001}


## Dependent variable

`y_bps = 1e4 * (cross / mid - 1)`, where `mid = (bid + ask) / 2` at message time.

This is the return from the prevailing mid at time *t* to the official closing cross —
the P&L of entering a position after 15:50 and exiting with an auction order at 16:00.

Predicting the *level* of `cross` is not a real problem: the current mid alone explains
~99.9% of its variance, so a model can score a near-perfect R² while containing no
information. All the predictable structure lives in the residual, and the residual is
the P&L. Basis points keep it comparable across a universe priced from $1 to $5,000.

In [25]:
clean_panel["mid"] = (clean_panel["bid"] + clean_panel["ask"]) / 2
clean_panel["y_bps"] = 1e4 * (clean_panel["cross"] / clean_panel["mid"] - 1)

clean_panel["y_bps"].describe(percentiles=[.01, .5, .99]).round(2)

count    3445083.00
mean          -1.74
std           25.12
min         -634.57
1%           -71.82
50%           -1.09
99%           67.50
max          736.84
Name: y_bps, dtype: float64

In [26]:
# the key validation: dispersion must shrink as the close approaches
bucket = pd.cut(clean_panel["secs_to_close"], [0, 60, 120, 300, 600],
                labels=["<1m", "1-2m", "2-5m", "5-10m"])
clean_panel.groupby(bucket, observed=True)["y_bps"].agg(["std", "mean", "count"]).round(3)

,std,mean,count
secs_to_close,,,
<1m,13.820,-0.094,626400
1-2m,18.365,-0.705,626400
2-5m,25.195,-2.804,1879145
5-10m,45.422,-0.680,313138


## Target validation

Dispersion of `y_bps` falls monotonically toward the close — 45.4 bps at 5–10 minutes
out, 13.8 bps inside the final minute. Uncertainty about the closing price has to
shrink as the auction approaches, so this is the check that the timestamp conversion,
the label broadcast and the target definition are all correct.

The target is stored **unwinsorized**. The widest symbol-days are microcaps and
leveraged single-stock ETFs, where a 400–700 bps move in ten minutes is ordinary. That
is heteroskedasticity, not corrupt data, and clipping it here would silently change
what the model is asked to predict.

Note that 90% of rows sit after 15:55: the message rate goes from one per 10s to one
per second, so half the window supplies 9% of the sample.

In [27]:
cols = ["ts", "date", "symbol", "secs_to_close",
        "bid", "ask", "bid_qty", "ask_qty", "mid",
        "side", "shares", "paired_shares", "ref_price", "near_price", "far_price",
        "adv", "open", "cross", "y_bps"]

out = clean_panel[cols].copy()
out.to_parquet(DATA / "clean.parquet", index=False)

print(f"wrote {len(out):,} rows x {out.shape[1]} cols | "
      f"{(DATA / 'clean.parquet').stat().st_size / 1e6:.0f} MB")
print(f"{out['date'].nunique()} days, {out['symbol'].nunique()} symbols")

wrote 3,445,083 rows x 19 cols | 156 MB
21 days, 560 symbols
